# 29 · Critical-case analysis of calibrated pig counting
**Project:** FaroPigSeg · First scientific manuscript  
**Google Colab runtime:** **CPU — GPU not required**.  
**Inputs:** `CERDOS_27_Resultados.zip` and `CERDOS_28_Resultados.zip`.  
**Outputs:** Publication-ready figures **in English** (PNG 600 DPI, vector PDF/SVG), Excel tables, provenance JSON, downloadable ZIP.

**Scientific safeguard:** The previously examined test set is analyzed descriptively. There is **no threshold retuning**, no model selection and no claim of independent external validation. Large overlaps between predicted masks are not automatically labelled as duplicate pigs. The notebook uses saved count predictions; it does not inspect the original pixels unless added in a separate, manually audited image-review stage.

In [ ]:
# 01 — Dependencies / CPU
import sys, subprocess, importlib.util
for pkg in ['pandas','openpyxl','matplotlib','numpy']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',pkg])
from pathlib import Path
import io, zipfile, json, hashlib, datetime
import pandas as pd, numpy as np, matplotlib.pyplot as plt
print('CPU analysis | Python',sys.version.split()[0])

In [ ]:
# 02 — Project directories and actual input files
try:
    from google.colab import drive,files
    drive.mount('/content/drive')
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
PROJECT=(Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖') if IN_COLAB else Path('/mnt/data'))
ROOTS=[PROJECT/'03_Resultados',PROJECT/'05_Exportaciones_ZIP',PROJECT,Path('/content'),Path('/mnt/data')]
def find_input(k):
    candidates=[]
    for folder in ROOTS:
        if folder.exists(): candidates.extend(folder.rglob(f'CERDOS_{k}_Resultados*.zip'))
    candidates=sorted(set(candidates),key=lambda x:x.stat().st_mtime,reverse=True)
    if candidates: return candidates[0]
    if IN_COLAB:
        print(f'Please upload CERDOS_{k}_Resultados.zip:')
        uploaded=files.upload()
        matches=[Path('/content')/q for q in uploaded if q.startswith(f'CERDOS_{k}_Resultados') and q.endswith('.zip')]
        if matches:return matches[0]
    raise FileNotFoundError(f'Missing CERDOS_{k}_Resultados.zip. Check 03_Resultados in Google Drive.')
P27=find_input(27);P28=find_input(28)
print('Experiment 27:',P27);print('Experiment 28:',P28)
def excel_from_zip(path):
    with zipfile.ZipFile(path) as z:
        workbooks=[x for x in z.namelist() if x.lower().endswith('.xlsx') and not x.startswith('__MACOSX/')]
        if len(workbooks)!=1: raise ValueError(f'Expected one Excel file in {path.name}; got {workbooks}')
        return pd.ExcelFile(io.BytesIO(z.read(workbooks[0])))
x27=excel_from_zip(P27); x28=excel_from_zip(P28)
print('27 sheets:',x27.sheet_names); print('28 sheets:',x28.sheet_names)

In [ ]:
# 03 — Data validation and strict replication of experiment 28
required={'Per_Image','Paired_Image_Deltas','Locked_Thresholds'}
if not required.issubset(x27.sheet_names):raise ValueError(f'Experiment 27 missing: {required-set(x27.sheet_names)}')
raw=pd.read_excel(x27,'Per_Image')
needed={'semilla','image_id','fecha','imagen','gt_conteo','strategy','threshold','predicted_count','absolute_error','signed_error'}
if not needed.issubset(raw.columns):raise ValueError(f'Experiment 27 columns missing: {needed-set(raw.columns)}')
assert set(raw.strategy)=={'Original (0.25)','Validation-calibrated'}
if raw[['semilla','image_id','strategy']].duplicated().any():raise ValueError('Duplicate image/seed/strategy')
if raw[list(needed)].isna().any().any():raise ValueError('Missing source values')
if not np.allclose(raw.absolute_error,(raw.predicted_count-raw.gt_conteo).abs()):raise ValueError('Absolute-error mismatch')
if not np.allclose(raw.signed_error,raw.predicted_count-raw.gt_conteo):raise ValueError('Signed-error mismatch')
locks=pd.read_excel(x27,'Locked_Thresholds')
seeds=sorted(raw.semilla.unique().tolist()); dates=sorted(raw.fecha.astype(str).unique().tolist())
if len(seeds)!=3 or len(dates)!=5:raise ValueError(f'Unexpected seeds or dates: {seeds}, {dates}')
for seed in seeds:
    rr=raw[raw.semilla==seed]
    th=locks.loc[locks.seed==seed,'threshold']
    if len(th)!=1:raise ValueError(f'Missing frozen threshold for {seed}')
    if not np.allclose(rr.loc[rr.strategy=='Validation-calibrated','threshold'],float(th.iloc[0])):raise ValueError('Frozen threshold changed')
    if not np.allclose(rr.loc[rr.strategy=='Original (0.25)','threshold'],.25):raise ValueError('Reference threshold changed')
pair=raw.pivot(index=['semilla','image_id','fecha','imagen','gt_conteo'],columns='strategy',values=['absolute_error','signed_error','predicted_count']).reset_index()
pair.columns=['_'.join(map(str,c)).strip('_') for c in pair.columns]
pair=pair.rename(columns={'absolute_error_Original (0.25)':'ae_original','absolute_error_Validation-calibrated':'ae_calibrated','signed_error_Original (0.25)':'bias_original','signed_error_Validation-calibrated':'bias_calibrated','predicted_count_Original (0.25)':'count_original','predicted_count_Validation-calibrated':'count_calibrated'})
pair['delta_ae']=pair.ae_calibrated-pair.ae_original
pair['outcome']=np.select([pair.delta_ae<0,pair.delta_ae>0],['Improved','Worsened'],default='Unchanged')
if pair[['semilla','image_id']].duplicated().any():raise ValueError('Unpaired or duplicated observations')
by_id=pair.pivot(index=['image_id','fecha','imagen','gt_conteo'],columns='semilla',values='delta_ae').reset_index()
by_id['all_seeds_worsened']=(by_id[seeds]>0).all(axis=1)
by_id['all_seeds_improved']=(by_id[seeds]<0).all(axis=1)
by_id['mean_delta_ae']=by_id[seeds].mean(axis=1)
critical=by_id[by_id.all_seeds_worsened].copy().sort_values('mean_delta_ae',ascending=False)
# Compare critical case count to original experiment 28 if audit table is included.
if 'Cross_Seed_Agreement' in x28.sheet_names:
    prev=pd.read_excel(x28,'Cross_Seed_Agreement')
    print('Cross-seed agreement sheet available:',len(prev))
print('Integrity PASS:',len(by_id),'images;',len(pair),'seed-image pairs; all-seed worsened:',len(critical))

In [ ]:
# 04 — Descriptive critical-case tables (no model tuning)
summary=pair.groupby(['semilla','outcome']).agg(n_images=('image_id','size'),mean_calibrated_error=('ae_calibrated','mean'),mean_delta_error=('delta_ae','mean')).reset_index()
summary['pct_images']=summary.n_images/summary.groupby('semilla').n_images.transform('sum')*100
by_date=pair.groupby(['semilla','fecha']).agg(n_images=('image_id','size'),mae_original=('ae_original','mean'),mae_calibrated=('ae_calibrated','mean'),worsened_pct=('delta_ae',lambda x:100*(x>0).mean()),residual_large_pct=('ae_calibrated',lambda x:100*(x>=3).mean())).reset_index()
# Flag for descriptive review only: >=3 pigs absolute error, an interpretable integer cutoff.
residual=pair[pair.ae_calibrated>=3].copy().sort_values(['ae_calibrated','delta_ae'],ascending=False)
by_id['worst_residual_pigs']=pair.groupby('image_id').ae_calibrated.max().reindex(by_id.image_id).values
by_id['n_seeds_large_residual']=pair.assign(large=pair.ae_calibrated>=3).groupby('image_id').large.sum().reindex(by_id.image_id).values
ranking=by_id.sort_values(['n_seeds_large_residual','worst_residual_pigs','mean_delta_ae'],ascending=False)
per_image_detail=pair.merge(by_id[['image_id','all_seeds_worsened','all_seeds_improved','n_seeds_large_residual']],on='image_id',validate='many_to_one')
print('Critical cases (all seeds worsen):',len(critical))
print('High residual error (>=3 pigs), seed-image records:',len(residual))
print(summary.round(2).to_string(index=False))

In [ ]:
# 05 — Publication-ready figures in English
OUT=PROJECT/'03_Resultados'/'CERDOS_29_Analisis_Casos_Criticos' if IN_COLAB else Path('/mnt/data/CERDOS_29_Analisis_Casos_Criticos')
FIG=OUT/'Figures';FIG.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.labelsize':11,'axes.titlesize':12,'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,'svg.fonttype':'none','savefig.dpi':600})
COL={'Improved':'#0072B2','Worsened':'#D55E00','Unchanged':'#999999'}
def save(name):
    plt.tight_layout()
    for ext in ['png','pdf','svg']:plt.savefig(FIG/f'{name}.{ext}',dpi=600,bbox_inches='tight')
    plt.close()
# Figure 1 - stacked percentage bars
fig,ax=plt.subplots(figsize=(7.0,4.2));left=np.zeros(len(seeds))
for result in ['Improved','Unchanged','Worsened']:
    values=[summary.loc[(summary.semilla==s)&(summary.outcome==result),'pct_images'].sum() for s in seeds]
    ax.barh(range(len(seeds)),values,left=left,label=result,color=COL[result],height=.55)
    left+=values
ax.set_yticks(range(len(seeds)),[f'Seed {s}' for s in seeds]);ax.set_xlabel('Share of evaluated images (%)');ax.set_xlim(0,100)
ax.set_title('Effect of calibration on individual counting errors');ax.legend(ncol=3,frameon=False,loc='upper center',bbox_to_anchor=(.5,-.14));ax.invert_yaxis();save('Fig29_01_Image_Level_Outcomes')
# Figure 2 - residual error distribution
fig,ax=plt.subplots(figsize=(7.0,4.3))
for j,s in enumerate(seeds):
    e=pair[pair.semilla==s].ae_calibrated.to_numpy()
    bins=np.arange(0,max(8,int(e.max())+2))
    hist=np.array([(e==k).mean()*100 for k in bins])
    ax.plot(bins,hist,marker=['o','s','^'][j],lw=1.8,label=f'Seed {s}')
ax.set_xlabel('Absolute calibrated counting error (pigs/image)');ax.set_ylabel('Images (%)');ax.set_title('Distribution of residual counting error');ax.legend(frameon=False);save('Fig29_02_Residual_Error_Distribution')
# Figure 3 - prevalence of worsening and residual error across dates
fig,ax=plt.subplots(figsize=(7.4,4.3))
for j,s in enumerate(seeds):
    p=by_date[by_date.semilla==s].sort_values('fecha')
    ax.plot(np.arange(len(p)),p.worsened_pct,marker=['o','s','^'][j],label=f'Seed {s}')
ax.set_xticks(range(len(dates)),[str(d)[5:] for d in dates],rotation=25);ax.set_xlabel('Acquisition date (2023)');ax.set_ylabel('Worsened images (%)')
ax.set_title('Cases worsened by calibration across dates');ax.legend(frameon=False);save('Fig29_03_Worsening_Across_Dates')
# Figure 4 - per-image cross-seed agreement
fig,ax=plt.subplots(figsize=(7,4.0))
counts=[int(by_id.all_seeds_improved.sum()),int(by_id.all_seeds_worsened.sum()),int((~by_id.all_seeds_improved & ~by_id.all_seeds_worsened).sum())]
labels=['All three improved','All three worsened','Mixed or unchanged']
bars=ax.barh(labels,counts,color=['#0072B2','#D55E00','#999999'])
for bar,c in zip(bars,counts):ax.text(c+2,bar.get_y()+bar.get_height()/2,str(c),va='center')
ax.set_xlim(0,max(counts)*1.17);ax.set_xlabel('Images');ax.set_title('Agreement of error changes across YOLO26 seeds');ax.invert_yaxis();save('Fig29_04_Cross_Seed_Agreement')
print('Publication figures:',[x.name for x in FIG.glob('*.pdf')])

In [ ]:
# 06 — Excel report, documented provenance, ZIP, automatic Colab download
excel_path=OUT/'CERDOS_29_Analisis_Casos_Criticos.xlsx'
with pd.ExcelWriter(excel_path,engine='openpyxl') as writer:
    summary.to_excel(writer,'Outcome_Summary',index=False)
    by_date.to_excel(writer,'By_Date',index=False)
    critical.to_excel(writer,'All_Seeds_Worsened',index=False)
    residual.to_excel(writer,'Large_Residual_Errors',index=False)
    ranking.to_excel(writer,'Image_Review_Priority',index=False)
    per_image_detail.to_excel(writer,'Per_Image_Detail',index=False)
    locks.to_excel(writer,'Frozen_Thresholds',index=False)
    pd.DataFrame([{'definition':'Worsened','rule':'Absolute error after calibration > original absolute error'},
                  {'definition':'Critical across seeds','rule':'All three seeds worsened on the same image'},
                  {'definition':'Large residual','rule':'Absolute calibrated counting error >= 3 pigs (descriptive flag only)'},
                  {'definition':'Data reuse','rule':'Previously inspected test set; descriptive analysis, not external validation'}]).to_excel(writer,'Methods_Notes',index=False)
prov={'experiment':29,'runtime':'CPU','utc_timestamp':datetime.datetime.now(datetime.timezone.utc).isoformat(),
      'input_27':P27.name,'sha256_27':hashlib.sha256(P27.read_bytes()).hexdigest(),
      'input_28':P28.name,'sha256_28':hashlib.sha256(P28.read_bytes()).hexdigest(),
      'n_images':int(len(by_id)),'n_seeds':len(seeds),'seeds':[int(x) for x in seeds],
      'n_dates':len(dates),'n_all_seeds_worsened':int(len(critical)),
      'frozen_thresholds':{str(int(row.seed)):float(row.threshold) for row in locks.itertuples()},
      'limitations':'Test set previously inspected; five acquisition dates, shared images across seeds; descriptive error characterization only.'}
(OUT/'CERDOS_29_Provenance.json').write_text(json.dumps(prov,indent=2),encoding='utf-8')
zip_path=OUT/'CERDOS_29_Resultados.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.rglob('*')):
        if f.is_file() and f != zip_path: z.write(f,f.relative_to(OUT))
print('Finished:',zip_path)
print('Critical images:',len(critical),'| figures:',len(list(FIG.glob('*.png'))),'| xlsx:',excel_path.name)
if IN_COLAB:
    files.download(str(zip_path))